# sfig8 — AUROC Gain Waterfall, Additional Tasks (Supplementary Fig. S-8)

**Data**: `analysis.csv` with all K values (phase0_v3)
**Tasks**: sex, age, apnea, depression, OSA (5 tasks not shown in main Fig. 6)
**Layout**: 3+2 mosaic (3 cols, last row centred)

Decomposes final AUROC into additive contributions:
1. **Base**: MeanPool @ 30s, K=1
2. **+Aggregation**: K=1→all at 30s
3. **+Context**: 30s→240m at MeanPool K=all
4. **+Architecture**: MeanPool→Transformer at 240m K=all
5. **=Final**: Transformer @ 240m K=all

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root ────────────────────────────────────────────────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis_all_k
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
METRIC = "mean_prob_auroc"
SPLIT  = "test"

# Load analysis.csv keeping ALL K values (needed for waterfall steps)
df_allk = load_analysis_all_k("phase0_v3", split=SPLIT)
print("Loaded rows:", len(df_allk))
print("K values sample:", sorted(df_allk["k"].astype(str).unique())[:8])

In [ ]:
# Panel label helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
_SUPP_TASKS = [
    "sex_binary",
    "age_class",
    "apnea_binary",
    "depression_extreme_binary",
    "osa_binary_apples_postqc",
]
_N_COLS = 3
_N_ROWS = (len(_SUPP_TASKS) + _N_COLS - 1) // _N_COLS   # = 2
_labels = [chr(97 + i) for i in range(len(_SUPP_TASKS))]
ROW_H   = 3.4 - 0.8   # match main_fig6_waterfall sizing

# 3+2 centred mosaic
_n_full = len(_SUPP_TASKS) // _N_COLS   # = 1
_mosaic = []
for _row in range(_n_full):
    _mosaic.append(list(_labels[_row * _N_COLS:(_row + 1) * _N_COLS]))
_last_row = list(_labels[_n_full * _N_COLS:])
_pad_left  = (_N_COLS - len(_last_row)) // 2
_pad_right = _N_COLS - len(_last_row) - _pad_left
_mosaic.append(["."] * _pad_left + _last_row + ["."] * _pad_right)

fig_supp, axd = plt.subplot_mosaic(
    _mosaic,
    figsize=(FULL_W * 2, _N_ROWS * ROW_H),
)

for _i, (_lbl, _task) in enumerate(zip(_labels, _SUPP_TASKS)):
    _ax = axd[_lbl]
    panels.waterfall_panel(_ax, df_allk, _task, metric=METRIC)
    _ax.set_title(TASK_LABEL[_task], fontsize=8)
    add_panel_label(_ax, f"({_lbl})")
    if _i % _N_COLS != 0:
        _ax.set_ylabel("")

fig_supp.tight_layout(h_pad=2.0, w_pad=1.5)
plt.show()

In [ ]:
# ── Run when figure looks good ────────────────────────────────────────────────
save_figure(fig_supp, FINAL_OUT, "sfig8_waterfall")
import shutil
shutil.copy(FINAL_OUT / "sfig8_waterfall.pdf",
            WORKSPACE_ROOT / "TBME_submission" / "sfig8_waterfall.pdf")
print("Saved + copied → TBME_submission/sfig8_waterfall.pdf")